- Эта тетрадь посвящена выгрузке таблиц xlxs из бд
На данный момент есть:
1. Таблица с базовой структурой разметки (там все)
```
    'case_iD': [],
    
    'sent_ID_RU': [],
    'sent_text_RU': [],
    'concept_unit_RU': [],
    
    'token_ID': [],
    'token_RU': [],
    'token_pos_RU': [],
    'type_RU': [],
    'connotation_RU': [],
    'sentiment_RU': [],
    'gram_structure_RU': [],
    
    тоже самое для английского
    
    'translation_shift': [],
    'cosine_sim_LaBSE': [],
    'shift_notes': [],
    'verified': []
```

2. Страницы для грамматики
3. Страница для коннотаций, типов, метафор и прочего

In [20]:
import os

import pandas as pd
from openpyxl import Workbook, load_workbook

import sqlite3
from pathlib import Path

In [21]:
DB_PATH = Path("../db/olfactory.db")
conn = sqlite3.connect(DB_PATH)
FILENAME = '../results/Разметка.xlsx'

## Export xlsx и листов

In [22]:
def create_excel_with_merges(df, filename, title, overwrite=False):
    """Автоматически использует все колонки из df в их порядке"""
    # Проверяем, существует ли файл
    if os.path.exists(filename):
        wb = load_workbook(filename)
    else:
        wb = Workbook()                  # Создаем новый файл
        if 'Sheet' in wb.sheetnames:     # Удаляем дефолтный лист "Sheet"
            wb.remove(wb['Sheet'])
    
    if overwrite and title in wb.sheetnames:
        wb.remove(wb[title])
        print(f"🔄 Лист '{title}' перезаписан")
    else:
        # Проверяем, существует ли лист с таким названием
        original_title = title
        counter = 1
        while title in wb.sheetnames:
            title = f"{original_title}_{counter}"
            counter += 1
        
        if title != original_title:
            print(f"⚠️ Лист '{original_title}' уже существует, создан '{title}'")
        
    # Создаем новый лист (теперь точно с уникальным названием)
    ws = wb.create_sheet(title=title)
    
    # # Заголовки с объединениями
    # ws['A1'] = '🇷🇺 РУССКИЙ ОРИГИНАЛ'
    # ws['C1'] = '🇬🇧 АНГЛИЙСКИЙ ПЕРЕВОД'
    # ws.merge_cells('A1:B1')
    # ws.merge_cells('C1:D1')
    
    # Подзаголовки - просто берем имена колонок из df
    for col_idx, col_name in enumerate(df.columns, start=1):
        ws.cell(row=1, column=col_idx, value=col_name)
    
    # Данные
    for r, (_, row) in enumerate(df.iterrows(), start=2):
        for col_idx, value in enumerate(row, start=1):
            ws.cell(row=r, column=col_idx, value=value)
    
    wb.save(filename)
    print(f"✅ Создан лист '{title}' в {filename} с колонками: {list(df.columns)}")

## Создание df для разных страниц

### Подключаемся к бд для заполнения

In [23]:
# информация о таблицах
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
for table in ['texts', 'translations', 'sentences', 'alignment']:
    count = pd.read_sql_query(f"SELECT COUNT(*) as cnt FROM {table}", conn)
    # print(f"{table}: {count.iloc[0]['cnt']}")

# tables

### Лист с полной разметкой

In [24]:
# Создаем пустой DataFrame с нужной структурой колонок
df = pd.DataFrame({
    'case_iD': [],
    
    'sent_ID_RU': [],
    'sent_text_RU': [],
    'concept_unit_RU': [],
    
    'token_ID': [],
    'token_RU': [],
    'token_pos_RU': [],
    'type_RU': [],
    'connotation_RU': [],
    'sentiment_RU': [],
    'gram_structure_RU': [],
    
    'sent_ID_EN': [],
    'sent_text_EN': [],
    'concept_unit_EN': [],
    
    'token_ID_EN': [],
    'token_EN': [],
    'token_pos_EN': [],
    'type_EN': [],
    'connotation_EN': [],
    'sentiment_EN': [],
    'gram_structure_EN': [],
    
    'translation_shift': [],
    'cosine_sim_LaBSE': [],
    'shift_notes': [],
    'verified': []
})

### Лист для грамматики

In [25]:
import pandas as pd
import numpy as np

# Получаем данные из БД через alignment (только парные предложения)
df_gr = pd.read_sql_query("""
    SELECT  
        ru.sentence_id as sent_ID_RU,
        ru.sentence as sent_text_RU,
        en.sentence_id as sent_ID_EN,
        en.sentence as sent_text_EN
    FROM alignment a
    JOIN sentences ru ON ru.sentence_id = a.sentence_ru_id AND ru.language = 'ru'
    JOIN sentences en ON en.sentence_id = a.sentence_en_id AND en.language = 'en'
    ORDER BY ru.sentence_id
""", conn)

print(f"✅ Загружено {len(df_gr)} пар предложений через alignment")

# Теперь добавляем нужные колонки в нужном порядке
df_gr = df_gr[[
    'sent_ID_RU', 'sent_text_RU', 
    'sent_ID_EN', 'sent_text_EN'
]].copy()

# Добавляем пустые колонки в том порядке, который вы хотите
df_gr['concept_unit_RU'] = ''
df_gr['gram_structure_RU'] = ''
df_gr['concept_unit_EN'] = ''
df_gr['gram_structure_EN'] = ''
df_gr['translation_shift'] = ''
df_gr['cosine_sim_LaBSE'] = np.nan
df_gr['shift_notes'] = ''
df_gr['verified'] = ''

# Переставляем колонки в нужном порядке
column_order = [
    'sent_ID_RU', 'sent_text_RU', 'concept_unit_RU', 'gram_structure_RU',
    'sent_ID_EN', 'sent_text_EN', 'concept_unit_EN', 'gram_structure_EN',
    'translation_shift', 'cosine_sim_LaBSE', 'shift_notes', 'verified'
]

df_gr = df_gr[column_order]
# df_gr

✅ Загружено 162 пар предложений через alignment


### Лист для типа, коннотации и тональности

In [ ]:
# Объединяем
df_tok = pd.DataFrame({
    'sent_ID_RU': df_gr['sent_ID_RU'],
    'sent_text_RU': df_gr['sent_text_RU'],

    'token_RU':'',

    'sentiment_RU': '',
    'type_RU': '',  # есть ли оценочное слово?
    'is_metaphor_RU': '',     # метафора или буквально?
    
    
    'sent_ID_EN': df_gr['sent_ID_EN'],
    'sent_text_EN': df_gr['sent_text_EN'],
    
    'token_EN':'',

    'sentiment_EN': '',
    'type_EN': '',  # есть ли оценочное слово?
    'is_metaphor_EN': '',     # метафора или буквально?
})



## Создание листов

In [27]:
create_excel_with_merges(df, FILENAME, 'Анализ')
create_excel_with_merges(df_gr, FILENAME, 'Разметка')
create_excel_with_merges(df_tok, FILENAME, 'Качество')

✅ Создан лист 'Анализ' в ../results/Разметка.xlsx с колонками: ['case_iD', 'sent_ID_RU', 'sent_text_RU', 'concept_unit_RU', 'token_ID', 'token_RU', 'token_pos_RU', 'type_RU', 'connotation_RU', 'sentiment_RU', 'gram_structure_RU', 'sent_ID_EN', 'sent_text_EN', 'concept_unit_EN', 'token_ID_EN', 'token_EN', 'token_pos_EN', 'type_EN', 'connotation_EN', 'sentiment_EN', 'gram_structure_EN', 'translation_shift', 'cosine_sim_LaBSE', 'shift_notes', 'verified']
✅ Создан лист 'Разметка' в ../results/Разметка.xlsx с колонками: ['sent_ID_RU', 'sent_text_RU', 'concept_unit_RU', 'gram_structure_RU', 'sent_ID_EN', 'sent_text_EN', 'concept_unit_EN', 'gram_structure_EN', 'translation_shift', 'cosine_sim_LaBSE', 'shift_notes', 'verified']
✅ Создан лист 'Качество' в ../results/Разметка.xlsx с колонками: ['sent_ID_RU', 'sent_text_RU', 'token_RU', 'sentiment_RU', 'has_evaluation_RU', 'is_metaphor_RU', 'sent_ID_EN', 'sent_text_EN', 'token_EN', 'sentiment_EN', 'has_evaluation_EN', 'is_metaphor_EN']


### Чтение, агрегации и прочие тесты

In [28]:
# 1. Читаем существующий Excel
df = pd.read_excel(FILENAME, sheet_name='Анализ', header=0)
# обратить внимание на header. Если строк с заголовками будет больше, то нучно ставить 1. Потому что заголовки на 2 строчке, а не 1. 
df 

,case_iD,sent_ID_RU,sent_text_RU,concept_unit_RU,token_ID,token_RU,token_pos_RU,type_RU,connotation_RU,sentiment_RU,...,token_EN,token_pos_EN,type_EN,connotation_EN,sentiment_EN,gram_structure_EN,translation_shift,cosine_sim_LaBSE,shift_notes,verified
